# Анализ доли автовыявляющихся ФП/СФП

Источник списка факторов: `autodetected_factors_from_photos.csv`.

Цель:
- посчитать долю этих факторов от всех выявленных за период 2023-2025;
- показать долю в целом и по сегментам;
- отдельно вывести коды из списка, которые не нашли совпадение в CRM.

In [ ]:
import warnings
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", None)

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"

LIST_FILE = "./autodetected_factors_from_photos.csv"

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith('.0'):
        s = s[:-2]
    s = s.lstrip('0') or '0'
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)


In [ ]:
# 1) Загружаем CRM в логике референсного отчета
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
df.columns = df.columns.str.strip()

df["inn"] = df["X_INN"].apply(normalize_inn)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()

if "VAL" in df.columns:
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()

df["TYPE_FP"] = df["TYPE_FP"].str.strip().replace({"FP": "ФП", "SFP": "СФП"})
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)
df = df[df["segment"].notna()].copy()

df["factor_code"] = df["NUMBER_FP_SFP"].astype(str).str.strip()
df = df.dropna(subset=["inn", "factor_code"]).copy()
df = df.drop_duplicates(subset=["inn", "factor_code", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()

print(f"Подготовлено CRM: {len(df):,} событий")
print(f"Уникальных ИНН: {df['inn'].nunique():,}")

In [ ]:
# 2) Загружаем список факторов из фото
lst = pd.read_csv(LIST_FILE)
lst["factor_code"] = lst["factor_code"].astype(str).str.strip()
lst["factor_type"] = lst["factor_type"].astype(str).str.strip()

factors = lst[["factor_type", "factor_code"]].drop_duplicates().copy()
print(f"Кодов в списке (уникальных по тип+код): {len(factors):,}")
display(factors.head(20))

In [ ]:
# 3) Матчим список к CRM и проверяем, что распознано
matched = df.merge(
    factors,
    left_on=["TYPE_FP", "factor_code"],
    right_on=["factor_type", "factor_code"],
    how="inner"
)

crm_codes = df[["TYPE_FP", "factor_code"]].drop_duplicates().rename(columns={"TYPE_FP": "factor_type"})
not_found = factors.merge(crm_codes, on=["factor_type", "factor_code"], how="left", indicator=True)
not_found = not_found[not_found["_merge"] == "left_only"].drop(columns=["_merge"])

print(f"Событий по списку (совпавших): {len(matched):,}")
print(f"Кодов из списка, не найденных в CRM: {len(not_found):,}")
if len(not_found) > 0:
    display(not_found.sort_values(["factor_type", "factor_code"]).reset_index(drop=True))

In [ ]:
# 4) Доли от всех выявленных: в целом и по сегментам
total_all = len(df)
total_fp = (df["TYPE_FP"] == "ФП").sum()
total_sfp = (df["TYPE_FP"] == "СФП").sum()

matched_all = len(matched)
matched_fp = (matched["TYPE_FP"] == "ФП").sum()
matched_sfp = (matched["TYPE_FP"] == "СФП").sum()

summary_overall = pd.DataFrame([
    ["Все события", total_all, matched_all, round(matched_all / total_all * 100, 2) if total_all else 0],
    ["Только ФП", total_fp, matched_fp, round(matched_fp / total_fp * 100, 2) if total_fp else 0],
    ["Только СФП", total_sfp, matched_sfp, round(matched_sfp / total_sfp * 100, 2) if total_sfp else 0],
], columns=["Срез", "Всего_выявлено", "Из_списка", "Доля_из_списка_%"])

display(summary_overall)

rows = []
for seg in SEG_ORDER:
    all_seg = df[df["segment"] == seg]
    m_seg = matched[matched["segment"] == seg]

    all_n = len(all_seg)
    m_n = len(m_seg)
    rows.append([seg, "Все события", all_n, m_n, round(m_n / all_n * 100, 2) if all_n else 0])

    all_fp_seg = (all_seg["TYPE_FP"] == "ФП").sum()
    m_fp_seg = (m_seg["TYPE_FP"] == "ФП").sum()
    rows.append([seg, "ФП", all_fp_seg, m_fp_seg, round(m_fp_seg / all_fp_seg * 100, 2) if all_fp_seg else 0])

    all_sfp_seg = (all_seg["TYPE_FP"] == "СФП").sum()
    m_sfp_seg = (m_seg["TYPE_FP"] == "СФП").sum()
    rows.append([seg, "СФП", all_sfp_seg, m_sfp_seg, round(m_sfp_seg / all_sfp_seg * 100, 2) if all_sfp_seg else 0])

summary_seg = pd.DataFrame(rows, columns=["Сегмент", "Срез", "Всего_выявлено", "Из_списка", "Доля_из_списка_%"])
display(summary_seg)

In [ ]:
# 5) Экспорт результатов
out_xlsx = f"{DATA_DIR}/Автовыявляющиеся_ФП_СФП_доли_2023_2025.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
    lst.to_excel(w, sheet_name="list_raw_from_photo", index=False)
    factors.to_excel(w, sheet_name="list_unique_type_code", index=False)
    summary_overall.to_excel(w, sheet_name="share_overall", index=False)
    summary_seg.to_excel(w, sheet_name="share_by_segment", index=False)
    if len(not_found) > 0:
        not_found.to_excel(w, sheet_name="codes_not_found", index=False)

print(f"Сохранено: {out_xlsx}")